# 🖼️ Aiscern — Image AI Detector v2
### Platform: **Kaggle T4 x2** | ~5h runtime
**Base model**: `google/vit-large-patch16-224` (ViT-Large — top image classification backbone)  
**Method**: LoRA fine-tuning on attention + MLP layers  
**Output push**: `saghi776/aiscern-image-detector`  
**Target**: ≥ 80% accuracy (F1 ≥ 0.80, AUC ≥ 0.87)

### Datasets (combined ~300k images)
| Dataset | Source | Samples | Notes |
|---|---|---|---|
| CIFAKE | bird-or-not/CIFAKE | 120k | CIFAR-10 real vs Stable Diffusion |
| ArtiFact | awsaf49/artifact | 60k | Multi-generator (SD, DALL-E, MJ) |
| GenImage | nyanko7/GenImage | 50k | 8 generators vs ImageNet real |
| AI vs Human Art | competitions/ai-vs-human-art | 40k | Midjourney vs real artwork |
| DeepFake Face | arnabdhar/DeepFake-Vs-Real-Faces | 30k | Face-focused deepfake |

### Wiring to Aiscern
Model loaded at weight inherited from ensemble in `hf-analyze.ts`:
```
MODELS.image_finetuned = 'saghi776/aiscern-image-detector'
```
Labels: `{0: "real", 1: "ai"}` — inference code uses `['ai','label_1','1']` as AI labels.

In [ ]:
!pip install -q transformers==4.41.0 datasets peft==0.11.0 accelerate==0.30.0 \
             evaluate scikit-learn huggingface_hub Pillow timm
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
print("✅ Packages installed")

In [ ]:
import os, torch, random, numpy as np
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token loaded")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN","")

BASE_MODEL       = "google/vit-large-patch16-224"
PUSH_REPO        = "saghi776/aiscern-image-detector"
CHECKPOINT_DIR   = "/kaggle/working/image-ckpt"
IMG_SIZE         = 224
BATCH_SIZE       = 16         # T4 16GB limit with ViT-Large
GRAD_ACCUM       = 4          # effective = 64
EPOCHS           = 5
LR               = 8e-5
LORA_R           = 16
LORA_ALPHA       = 32
SAMPLES_PER_CLASS= 55000
SEED             = 42

random.seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device} | {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

In [ ]:
# ── LOAD DATASETS ─────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets

dataset_configs = [
    ("datasets/CIFAKE",             "train", lambda r: {"image": r["image"], "label": r["label"]}),  # 0=real,1=fake
    ("awsaf49/artifact",            "train", lambda r: {"image": r["image"], "label": 0 if r.get("label","real")=="real" else 1}),
]

def norm_image_label(row, label_field="label", ai_vals=(1,"ai","fake","generated","sd","midjourney","dalle")):
    v = str(row.get(label_field,"")).lower().strip()
    return 1 if (v in [str(a) for a in ai_vals] or any(str(a) in v for a in ai_vals if len(str(a))>3)) else 0

all_real, all_ai = [], []

# CIFAKE (most reliable — 120k balanced)
print("Loading CIFAKE...")
try:
    cf = load_dataset("jungwoo-ha/CIFAKE", split="train", trust_remote_code=True)
    for row in cf:
        entry = {"image": row["image"], "label": row["label"]}
        if row["label"] == 0: all_real.append(entry)
        else:                 all_ai.append(entry)
    print(f"  CIFAKE: real={len(all_real):,} ai={len(all_ai):,}")
except Exception as e:
    print(f"  CIFAKE failed ({e}), trying bird-or-not/CIFAKE...")
    try:
        cf = load_dataset("bird-or-not/CIFAKE", split="train", trust_remote_code=True)
        for row in cf:
            lbl = norm_image_label(row)
            if lbl==0: all_real.append({"image":row["image"],"label":0})
            else:      all_ai.append({"image":row["image"],"label":1})
        print(f"  CIFAKE alt: real={len(all_real):,} ai={len(all_ai):,}")
    except Exception as e2: print(f"  Both CIFAKE variants failed: {e2}")

# DeepFake faces
print("Loading DeepFake Faces...")
try:
    df = load_dataset("arnabdhar/DeepFake-Vs-Real-Faces", split="train", trust_remote_code=True)
    prev_r, prev_a = len(all_real), len(all_ai)
    for row in df.select(range(min(len(df),30000))):
        lbl = 1 if "fake" in str(row.get("label","")).lower() else 0
        if lbl==0: all_real.append({"image":row["image"],"label":0})
        else:      all_ai.append({"image":row["image"],"label":1})
    print(f"  DeepFake: +real={len(all_real)-prev_r:,} +ai={len(all_ai)-prev_a:,}")
except Exception as e: print(f"  DeepFake failed: {e}")

# AI vs Human Art
print("Loading AI vs Human Art...")
try:
    art = load_dataset("competitions/ai-vs-human-art", split="train", trust_remote_code=True)
    prev_r, prev_a = len(all_real), len(all_ai)
    for row in art.select(range(min(len(art),40000))):
        lbl = int(row.get("label",0))
        if lbl==0: all_real.append({"image":row["image"],"label":0})
        else:      all_ai.append({"image":row["image"],"label":1})
    print(f"  Art: +real={len(all_real)-prev_r:,} +ai={len(all_ai)-prev_a:,}")
except Exception as e: print(f"  Art failed: {e}")

print(f"\n📊 Total — Real: {len(all_real):,} | AI: {len(all_ai):,}")

In [ ]:
# ── BALANCE + TRANSFORM ──────────────────────────────────────────
from PIL import Image
from torchvision import transforms

real_sample = random.sample(all_real, min(len(all_real), SAMPLES_PER_CLASS))
ai_sample   = random.sample(all_ai,   min(len(all_ai),   SAMPLES_PER_CLASS))
combined    = real_sample + ai_sample
random.shuffle(combined)

# Image augmentation pipeline — increases robustness to compression/resize artifacts
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE+32, IMG_SIZE+32)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
])

split_idx = int(len(combined) * 0.90)
train_raw = combined[:split_idx]
val_raw   = combined[split_idx:]
print(f"Train: {len(train_raw):,} | Val: {len(val_raw):,}")

class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, rows, transform):
        self.rows = rows
        self.transform = transform
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        img = row["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img) if hasattr(img,"__array_interface__") else Image.open(img)
        img = img.convert("RGB")
        return {"pixel_values": self.transform(img), "labels": torch.tensor(row["label"])}

train_ds = ImageDataset(train_raw, train_transform)
val_ds   = ImageDataset(val_raw,   val_transform)
print(f"✅ Datasets ready")

In [ ]:
# ── MODEL + LoRA ─────────────────────────────────────────────────
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model, TaskType

processor = ViTImageProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)
model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "real", 1: "ai"},   # hf-analyze.ts: parseHFText looks for 'ai' label
    label2id={"real": 0, "ai": 1},
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    # ViT attention layers
    target_modules=["query","key","value","dense"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── TRAINING ─────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    labels       = torch.stack([b["labels"]       for b in batch])
    return {"pixel_values": pixel_values, "labels": labels}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1":       f1_score(labels, preds),
        "roc_auc":  roc_auc_score(labels, probs),
    }

args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("🚀 Training ViT-Large + LoRA...")
trainer.train()

In [ ]:
# ── EVALUATE + PUSH ──────────────────────────────────────────────
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

results  = trainer.evaluate()
acc = results.get("eval_accuracy", 0)
f1  = results.get("eval_f1", 0)
auc = results.get("eval_roc_auc", 0)
print(f"\n🏆 Accuracy: {acc*100:.2f}% | F1: {f1*100:.2f}% | AUC: {auc*100:.2f}%")
print(f"{'✅ PASSED' if acc>=0.80 else '⚠  Below 80% — add more diverse AI generator samples'}")

# Confusion matrix
preds_out = trainer.predict(val_ds)
preds  = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["Real","AI"], yticklabels=["Real","AI"], cmap="Greens")
plt.title(f"Image Detector — Acc={acc*100:.1f}%")
plt.tight_layout(); plt.show()
print(classification_report(labels, preds, target_names=["real","ai"]))

# Push merged model
print("\nMerging LoRA + pushing to HF...")
merged = model.merge_and_unload()
merged.push_to_hub(PUSH_REPO, token=HF_TOKEN)
processor.push_to_hub(PUSH_REPO, token=HF_TOKEN)
print(f"✅ Live at https://huggingface.co/{PUSH_REPO}")
print("   hf-analyze.ts MODELS.image_finetuned points here automatically.")